# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata object and print overview
meta = dataset.metadata

print(f"Dataset Name : {meta.name}")
print(f"Description : {meta.description}")
print(f"Published   : {meta.datePublished}")
print(f"License     : {meta.license}")
print(f"Version     : {meta.version}")


## 2. Data Overview
Review available record sets, fields, and their IDs. All schemas and entities are referenced via their `@id` field as per Croissant standard.

In [ ]:
# List all record sets in the dataset
print("Available Record Sets (by @id):")
record_sets = list(dataset.metadata.record_sets)
for record_set in record_sets:
    print(f"  - @id: {record_set.id} | Name: {record_set.name}")

# For each record set, show fields and columns (by @id):
for record_set in record_sets:
    print(f"\nFields in Record Set '{record_set.name}' (@id: {record_set.id}):")
    for field in record_set.fields:
        print(f"   • Field: {field.name}  (@id: {field.id})  | DataType: {field.data_type}")
        # Each field typically refers to a column:
        if field.column is not None:
            print(f"     ↳ Column @id: {field.column.id} ({field.column.name})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Set up extraction using the `@id` of the selected record set and its main fields.

In [ ]:
# Choose the principal record set for data extraction
# Typically named like 'ProteinTable' or similar, but we identify by @id
# For demonstration, we take the first record set listed:

main_record_set = record_sets[0]

print(f"Extracting data from Record Set: {main_record_set.name} (@id: {main_record_set.id})")

# List all available field @id
field_ids = [field.id for field in main_record_set.fields]
print("Fields @id:", field_ids)

records = list(dataset.records(record_set=main_record_set.id))
df = pd.DataFrame(records)
print(f"Columns loaded: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Note:** We reference fields by their `@id`. Adjust the example below to reference an actual numeric field and grouping field as shown in the overview.

In [ ]:
# Example: Filter by a numeric field such as peptide count or molecular weight
# Determine a numeric field to analyze (e.g., 'cr:peptide_count' or actual field @id from previous cell)
# For demonstration, we select the first numeric column we find

numeric_field_id = None
for field in main_record_set.fields:
    # Use field.data_type to detect numeric
    # Croissant field.type often like 'schema:Integer' or 'schema:Float', etc.
    if field.data_type.lower() in ['integer', 'float', 'number', 'double', 'schema:integer', 'schema:float', 'schema:number']:
        numeric_field_id = field.id
        break
if numeric_field_id is None:
    numeric_field_id = df.select_dtypes('number').columns[0]  # fallback by dtype

print(f"Using numeric field for EDA: {numeric_field_id}")

# Filtering out records with low or zero values
threshold = df[numeric_field_id].mean() if numeric_field_id in df else 0
filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id in df else df
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
if numeric_field_id in filtered_df:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (choose one with low cardinality)
group_field_id = None

for field in main_record_set.fields:
    # group by string/categorical
    if field.data_type.lower() in ['string', 'text', 'schema:text']:
        if field.id != numeric_field_id and field.id in df.columns:
            if df[field.id].nunique() < 10:
                group_field_id = field.id
                break
if not group_field_id:
    # fallback: just pick first non-numeric field
    for col in df.columns:
        if col != numeric_field_id:
            if df[col].dtype == 'object' and df[col].nunique() < 10:
                group_field_id = col
                break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (showing group means):")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
if numeric_field_id in df:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Boxplot by group (if group_field exists)
if group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(12,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!-- 
Key findings:
- The dataset provides mass spectrometry records of human extracellular vesicles with detailed fields on protein properties.
- Record sets, fields, and columns are referenced strictly by their `@id` for best practice FAIR, as recommended by the Croissant schema.
- The top numeric field in this record set shows [insert observation here after running].
- Exploratory plots allow quick checks for outliers or sample-specific shifts.
- Grouping and summary statistics can guide further downstream analyses such as biomarker discovery or batch correction.
-->

_You can now extend this notebook to perform additional domain-specific analyses or machine learning tasks specific to proteomics or your research question!_